In [3]:
# instala Java e PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q pyspark==3.5.1

import os, glob
# encontra automaticamente a pasta do Java instalada
os.environ["JAVA_HOME"] = sorted(glob.glob("/usr/lib/jvm/java-*-openjdk-amd64"))[-1]
os.environ.pop("SPARK_HOME", None)   # usa o Spark do pip, não o .tgz
print("JAVA_HOME =", os.environ["JAVA_HOME"])

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-lts/openjdk-11-jre-headless_11.0.32.1%2b1-1ubuntu1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.82 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-lts/openjdk-11-jdk-headless_11.0.32.1%2b1-1ubuntu1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.82 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64


In [2]:
!java -version
!ls /usr/lib/jvm

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
java-1.21.0-openjdk-amd64  java-21-openjdk-amd64


In [5]:
# baixa o dataset e prepara o df
!wget -q -nc https://raw.githubusercontent.com/bensadeghi/pyspark-churn-prediction/master/data/churn-bigml-20.csv

df = spark.read.csv('churn-bigml-20.csv', header=True, inferSchema=True)
df = df.dropDuplicates()
df = df.withColumnRenamed('Account length', 'Account')
df = df.drop('Number vmail messages')
df.createOrReplaceTempView("temp")

df.count()

667

# ATIVIDADE PRÁTICA

In [6]:
#média de chamadas por código de área (Total day calls)
df.groupBy('Area code').agg({'Total day calls': 'mean'}).orderBy('Area code').show()

+---------+--------------------+
|Area code|avg(Total day calls)|
+---------+--------------------+
|      408|    99.3905325443787|
|      415|  101.92878338278932|
|      510|  100.48447204968944|
+---------+--------------------+



In [7]:
#média de minutos por código de área (Total day minutes)
df.groupBy('Area code').agg({'Total day minutes': 'mean'}).orderBy('Area code').show()

+---------+----------------------+
|Area code|avg(Total day minutes)|
+---------+----------------------+
|      408|     175.4520710059171|
|      415|    181.42225519287842|
|      510|     185.7248447204969|
+---------+----------------------+



In [8]:
#registros com Total night calls menor que 55
noturnas = df.filter(df['Total night calls'] < 55)
noturnas.show(truncate=False)

+-----+-------+---------+------------------+---------------+-----------------+---------------+----------------+-----------------+---------------+----------------+-------------------+-----------------+------------------+------------------+----------------+-----------------+----------------------+-----+
|State|Account|Area code|International plan|Voice mail plan|Total day minutes|Total day calls|Total day charge|Total eve minutes|Total eve calls|Total eve charge|Total night minutes|Total night calls|Total night charge|Total intl minutes|Total intl calls|Total intl charge|Customer service calls|Churn|
+-----+-------+---------+------------------+---------------+-----------------+---------------+----------------+-----------------+---------------+----------------+-------------------+-----------------+------------------+------------------+----------------+-----------------+----------------------+-----+
|ID   |113    |415      |No                |Yes            |183.8            |102          